# Pallas EP MoE Forward Kernel — Workshop

**Goal**: Write a fused Expert-Parallel MoE forward kernel from scratch in Pallas/Mosaic.

**Pacing**: ~2 hours per Day. You can stop at any Day boundary and pick up next session.

**How to use this notebook**:
- Each exercise has a `# YOUR CODE HERE` cell — implement it yourself first.
- Solution cells below are clearly marked `# SOLUTION — peek if stuck`. They are collapsed by default.
- Run cells top to bottom; later exercises build on earlier ones.

**Target config** (what the full kernel will run on):
```
DSv3 671B, 4×4×16 bodaborg cluster
EP=32, FSDP=16, TP=1
T_fsdp=65536, D=7168, E=256, E_local=8, F_shard=128, K=8
```

---

## Setup

In [1]:
# Run this cell first every session.
# Activate xprof venv before launching: source ~/xdb/.xprof/bin/activate

import functools
import numpy as np
import jax
import jax.numpy as jnp
from jax import lax
from jax.experimental import pallas as pl
from jax.experimental.pallas import tpu as pltpu
from jax.experimental.shard_map import shard_map
from jax.sharding import PartitionSpec as P

print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")

/tmp/ipykernel_2337813/4260370075.py:11: DeprecationWarning: jax.experimental.shard_map is deprecated in v0.8.0. Used jax.shard_map instead.
  from jax.experimental.shard_map import shard_map


JAX version: 0.9.2
Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0), TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0)]


---
# Day 0 — Pallas Fundamentals (~2 hrs)

**Goal**: Understand `pallas_call`, BlockSpec, and async local DMA.

Before diving into MoE, you need to internalize Pallas's core abstraction:
you control every data movement explicitly. Unlike JAX where you write `y = x @ w`
and XLA tiles it, in Pallas you write: "bring tile of x to VMEM, bring tile of w to VMEM,
compute, write result back."

## Memory hierarchy recap

```
HBM (~100 GB, ~1 TB/s)   ←→ DMA ←→   VMEM (~32 MB, ~10 TB/s)
                                              ↕
                                       SMEM (routing tables, 1-D only)
```

- **VMEM**: where all compute happens. Holds weight tiles, activation tiles, accumulators.
- **SMEM**: for index tables and metadata. Must be 1-D arrays.
- **HBM**: off-chip, you never compute here. Move data in/out via async DMA.

## The grid abstraction

A Pallas kernel runs a body function `num_bt` times (one per "block" of tokens).
Mosaic compiles the body **once** and instantiates it `num_bt` times.
Inside the body, `pl.program_id(0)` gives the current iteration index.

## Exercise 0.1 — Hello Pallas: copy tokens HBM→VMEM→HBM

Write a kernel that reads a tile of tokens from HBM, stores them in a VMEM scratch buffer,
then writes them back out — a simple identity copy with an intermediate VMEM stop.

**Why**: This forces you to understand DMA semaphores and the VMEM Ref pattern before
adding any compute complexity.

In [2]:
# Exercise 0.1 — Identity copy via VMEM
#
# Given: tokens (T, D) in HBM
# Return: tokens (T, D) — unchanged, but passed through VMEM
#
# Shapes for this exercise:
T, D = 64, 128
BT = 8   # tile size: process BT rows per grid step

def copy_kernel(
    tokens_ref,   # HBM ref: full (T, D) — slice per step via explicit DMA
    out_ref,      # HBM ref: full (T, D) — write per step via explicit DMA
    vmem_buf,     # VMEM scratch: (BT, D) — compute in-place here
    sem_ref,      # DMA semaphore
):
    bt_id = pl.program_id(0)

    # DMA in: HBM[bt_id*BT : (bt_id+1)*BT, :] → vmem_buf
    dma_in = pltpu.make_async_copy(
        src_ref=tokens_ref.at[pl.ds(bt_id * BT, BT), :],
        dst_ref=vmem_buf,
        sem=sem_ref,
    )
    dma_in.start()
    dma_in.wait()

    # DMA out: vmem_buf → HBM out[bt_id*BT : (bt_id+1)*BT, :]
    dma_out = pltpu.make_async_copy(
        src_ref=vmem_buf,
        dst_ref=out_ref.at[pl.ds(bt_id * BT, BT), :],
        sem=sem_ref,
    )
    dma_out.start()
    dma_out.wait()


def run_copy_kernel(tokens):
    return pl.pallas_call(
        copy_kernel,
        grid_spec=pltpu.PrefetchScalarGridSpec(
            num_scalar_prefetch=0,
            grid=(T // BT,),
            # HBM BlockSpec: trivial index_map (all zeros), block_shape = full array shape.
            # Non-trivial index_map with HBM raises ValueError — the auto-store path
            # only works for VMEM (default).  With HBM we slice explicitly via DMA.
            in_specs=[pl.BlockSpec((T, D), lambda i: (0, 0),
                                   memory_space=pltpu.MemorySpace.HBM)],
            out_specs=pl.BlockSpec((T, D), lambda i: (0, 0),
                                   memory_space=pltpu.MemorySpace.HBM),
            scratch_shapes=[
                pltpu.VMEM((BT, D), jnp.bfloat16),
                pltpu.SemaphoreType.DMA,
            ],
        ),
        out_shape=jax.ShapeDtypeStruct((T, D), jnp.bfloat16),
    )(tokens)


# Test
key = jax.random.PRNGKey(0)
tokens_np = jax.random.normal(key, (T, D), dtype=jnp.bfloat16)
result = run_copy_kernel(tokens_np)
np.testing.assert_allclose(result, tokens_np, rtol=0, atol=0)
print("[PASS] copy kernel")

[PASS] copy kernel


In [3]:
# SOLUTION — peek if stuck
#
# Two bugs in the naive approach:
#   1. in_specs=[None] was removed in JAX 0.9.2 — raises ValueError.
#   2. HBM in + VMEM out (auto-store) → BoundsCheck at last grid step.
#
# Fix: use memory_space=pltpu.MemorySpace.HBM for BOTH in and out,
# with block_shape = full array shape and trivial index_map (all zeros).
# Then slice explicitly in the kernel body via pl.ds.
#
# Shapes for this exercise:
T, D = 64, 128
BT = 8   # tile size: process BT rows per grid step

def _copy_kernel_sol(
    tokens_hbm,   # HBM ref: full (T, D) array
    out_hbm,      # HBM ref: full (T, D) array
    vmem_buf,     # VMEM scratch: (BT, D)
    sem_ref,      # DMA semaphore
):
    bt_id = pl.program_id(0)

    # DMA in: HBM[bt_id*BT : (bt_id+1)*BT, :] → vmem_buf
    dma_in = pltpu.make_async_copy(
        src_ref=tokens_hbm.at[pl.ds(bt_id * BT, BT), :],
        dst_ref=vmem_buf,
        sem=sem_ref,
    )
    dma_in.start()
    dma_in.wait()

    # DMA out: vmem_buf → HBM out[bt_id*BT : (bt_id+1)*BT, :]
    dma_out = pltpu.make_async_copy(
        src_ref=vmem_buf,
        dst_ref=out_hbm.at[pl.ds(bt_id * BT, BT), :],
        sem=sem_ref,
    )
    dma_out.start()
    dma_out.wait()


def _run_copy_kernel_sol(tokens):
    return pl.pallas_call(
        _copy_kernel_sol,
        grid_spec=pltpu.PrefetchScalarGridSpec(
            num_scalar_prefetch=0,
            grid=(T // BT,),
            in_specs=[pl.BlockSpec((T, D), lambda i: (0, 0),
                                   memory_space=pltpu.MemorySpace.HBM)],
            out_specs=pl.BlockSpec((T, D), lambda i: (0, 0),
                                   memory_space=pltpu.MemorySpace.HBM),
            scratch_shapes=[
                pltpu.VMEM((BT, D), jnp.bfloat16),
                pltpu.SemaphoreType.DMA,
            ],
        ),
        out_shape=jax.ShapeDtypeStruct((T, D), jnp.bfloat16),
    )(tokens)


key = jax.random.PRNGKey(0)
tokens_np = jax.random.normal(key, (T, D), dtype=jnp.bfloat16)
result = _run_copy_kernel_sol(tokens_np)
np.testing.assert_allclose(np.array(result), np.array(tokens_np), rtol=0, atol=0)
print("[PASS] copy kernel")


[PASS] copy kernel


## Exercise 0.2 — BlockSpec index maps and pl.ds

Understand how BlockSpec slices HBM. The index_map controls which tile to load per grid step.

**Task**: Write a kernel that, for each tile of tokens, multiplies each row by a scalar
from a scale vector `scales: (T,)`. The scales vector maps token index → per-token scalar.

**Key challenge**: `scales` needs to be accessed at `scales[bt_id * BT : (bt_id+1) * BT]`.
Use `pl.ds(start, size)` for a dynamic slice within VMEM.

In [4]:
# Exercise 0.2 — per-token scale
T, D = 1024, 128
BT = 256   # tile size: process BT rows per grid step
key = jax.random.PRNGKey(0)
tokens_np = jax.random.normal(key, (T, D), dtype=jnp.bfloat16)


def scale_kernel(
    tokens_ref,   # (BT, D) — BlockSpec loads BT rows per step
    scales_ref,   # (T,) — load ALL scales every step (simplification)
    out_ref,
    vmem_buf,
    sem_ref,
):
    bt_id = pl.program_id(0)
    # dma_in = pltpu.make_async_copy(
    #     src_ref=tokens_ref,at[pl.ds(bt_id * BT, BT)],
    #     dst_ref=vmem_buf,
    #     sem=sem_ref,
    # )
    # dma_in.start()
    # dma_in.wait()
    scale = scales_ref[pl.ds(bt_id * BT, BT)]
    #scale_2d = lax.broadcast_in_dim(scale, (BT, D), (0,))
    #vmem_buf[...] = tokens_ref [...] * scale_2d
    #for i in range(BT):
    #    vmem_buf[i, :] = tokens_ref[i, :] * scale[i]
    scale_f32 = scale.astype(jnp.float32)                                             
    for i in range(BT):
        vmem_buf[i, :] = (tokens_ref[i, :] * scale_f32[i]).astype(jnp.bfloat16)
    dma_out = pltpu.make_async_copy(
        src_ref=vmem_buf,
        dst_ref=out_ref.at[pl.ds(bt_id * BT, BT), :],
        sem=sem_ref,
    )
    dma_out.start()
    dma_out.wait()
    # YOUR CODE HERE
    # 1. Load tokens into vmem_buf via async DMA (start + wait)
    # 2. Extract the BT scales for this tile: scales_ref[bt_id*BT : (bt_id+1)*BT]
    #    Use pl.ds(bt_id * BT, BT) for the dynamic slice
    # 3. Multiply each row of vmem_buf by its scale
    #    Hint: avoid [:, None] — build a (BT, D) weight matrix or broadcast carefully
    # 4. Write result to out_ref
   # pass

def run_scale_kernel(tokens, scale):
    return pl.pallas_call(
        scale_kernel,
        grid_spec=pltpu.PrefetchScalarGridSpec(
            num_scalar_prefetch=0,
            grid =(T // BT,),
            in_specs=[
                pl.BlockSpec((BT, D), lambda i: (i, 0)),
                pl.BlockSpec((T, ), lambda i: (0,))
            ],               
            out_specs=pl.BlockSpec((T, D), lambda i: (0, 0), memory_space=pltpu.MemorySpace.HBM),
            scratch_shapes=[
                pltpu.VMEM((BT, D), jnp.bfloat16),
                pltpu.SemaphoreType.DMA,
            ]
        ),
        out_shape=jax.ShapeDtypeStruct((T, D), jnp.bfloat16)
    )(tokens, scale)
        
#Test (uncomment to run once implemented)
scales = jax.random.uniform(jax.random.PRNGKey(1), (T,), dtype=jnp.bfloat16)
result = run_scale_kernel(tokens_np, scales)
expected = tokens_np * scales[:, None]
np.testing.assert_allclose(result, expected, rtol=1e-2)
print("[PASS] scale kernel")

[PASS] scale kernel


In [5]:
# SOLUTION — peek if stuck
#
# Why the naive approaches fail on TPU v4 / JAX 0.9.2:
#
#   1. VMEM BlockSpec with non-trivial index_map (lambda i: (i*BT, 0)) causes
#      RuntimeUnexpectedCoreHalt on v4 — only constant/trivial (0,...) index_maps
#      work.  Fix: use HBM BlockSpec (memory_space=HBM) with block_shape == full
#      array shape and trivial index_map, then slice explicitly via DMA.
#
#   2. scales_ref[pl.ds(bt_id*BT, BT)] on a VMEM Ref → dynamic_slice_p, which
#      is NotImplementedError in Pallas TC on v4.
#
#   3. lax.fori_loop(0, BT, ...) → Mosaic DMA BoundsCheck at the last grid step.
#
#   4. DMA chunk < 128 bytes (e.g. 8 float32 = 32 bytes) violates TPU DMA
#      alignment.  To DMA the BT=8 scales we must load ALL T scales instead:
#      T * sizeof(float32) = 64*4 = 256 bytes ≥ 128 ✓.
#
# Working approach:
#   • HBM BlockSpec (trivial index_map) for tokens and scales.
#   • Explicit DMA in: tokens tile (BT,D) and ALL scales (T,).
#   • Fully unrolled Python for loop (BT=8 iterations, no lax.fori_loop):
#       each iteration extracts scales_all[bt_off+j] using a 1D mask on (T,),
#       then accumulates row j into scale_mat (BT,D) using a row selector.
#   • No [:, None], no reshape, no dynamic_slice.

def _scale_kernel_sol(
    tokens_hbm,   # HBM: full (T, D) — explicit DMA by tile
    scales_hbm,   # HBM: full (T,)  — DMA all T scales each step
    out_hbm,      # HBM: full (T, D) — explicit DMA out
    vmem_buf,     # VMEM scratch: (BT, D) bfloat16
    vmem_scales,  # VMEM scratch: (T,)  float32
    sem_ref,      # DMA semaphore
):
    bt_id = pl.program_id(0)

    # DMA in: tokens tile → vmem_buf
    dma_tok = pltpu.make_async_copy(
        tokens_hbm.at[pl.ds(bt_id * BT, BT), :], vmem_buf, sem_ref)
    dma_tok.start(); dma_tok.wait()

    # DMA in: ALL T scales → vmem_scales  (BT*4=32 bytes too small; load T*4=256 bytes instead)
    dma_sc = pltpu.make_async_copy(
        scales_hbm.at[pl.ds(0, T)], vmem_scales, sem_ref)
    dma_sc.start(); dma_sc.wait()

    tokens_f32 = vmem_buf[...].astype(jnp.float32)   # (BT, D)
    scales_all = vmem_scales[...]                      # (T,) float32

    bt_off   = bt_id * BT
    T_iota   = lax.broadcasted_iota(jnp.int32, (T,), 0)    # (T,): 0..T-1
    row_iota = lax.broadcasted_iota(jnp.int32, (BT, D), 0) # (BT,D): row index

    # Build scale_mat (BT, D) — fully unrolled (no lax.fori_loop: causes BoundsCheck).
    # Row j: extract scalar scales_all[bt_off+j] with a 1D mask; broadcast via row selector.
    scale_mat = jnp.zeros((BT, D), jnp.float32)
    for j in range(BT):
        t_mask = (1 - jnp.minimum(jnp.abs(T_iota - bt_off - j), 1)).astype(jnp.float32)
        s_j    = jnp.sum(scales_all * t_mask)                                        # scalar
        row_eq = (1 - jnp.minimum(jnp.abs(row_iota - j), 1)).astype(jnp.float32)    # (BT,D)
        scale_mat = scale_mat + row_eq * s_j

    vmem_buf[...] = (tokens_f32 * scale_mat).astype(jnp.bfloat16)

    # DMA out: vmem_buf → HBM output tile
    dma_out = pltpu.make_async_copy(
        vmem_buf, out_hbm.at[pl.ds(bt_id * BT, BT), :], sem_ref)
    dma_out.start(); dma_out.wait()


def _run_scale_kernel_sol(tokens, scales):
    return pl.pallas_call(
        _scale_kernel_sol,
        grid_spec=pltpu.PrefetchScalarGridSpec(
            num_scalar_prefetch=0,
            grid=(T // BT,),
            in_specs=[
                pl.BlockSpec((T, D), lambda i: (0, 0), memory_space=pltpu.MemorySpace.HBM),
                pl.BlockSpec((T,),   lambda i: (0,),   memory_space=pltpu.MemorySpace.HBM),
            ],
            out_specs=pl.BlockSpec((T, D), lambda i: (0, 0), memory_space=pltpu.MemorySpace.HBM),
            scratch_shapes=[
                pltpu.VMEM((BT, D), jnp.bfloat16),
                pltpu.VMEM((T,), jnp.float32),
                pltpu.SemaphoreType.DMA,
            ],
        ),
        out_shape=jax.ShapeDtypeStruct((T, D), jnp.bfloat16),
    )(tokens, scales)


scales = jax.random.uniform(jax.random.PRNGKey(1), (T,), dtype=jnp.float32)
result = _run_scale_kernel_sol(tokens_np, scales)
expected = (np.array(tokens_np).astype(np.float32) * np.array(scales)[:, None]).astype(np.float16)
np.testing.assert_allclose(np.array(result), expected, rtol=1e-2)
print("[PASS] scale kernel")


[PASS] scale kernel


## Exercise 0.3 — scalar_prefetch and SMEM

Some per-step metadata (routing tables, offsets) should live in SMEM rather than
being reloaded from HBM each step. `scalar_prefetch` args are loaded into SMEM
automatically before each grid step.

**Task**: Write a kernel that uses a scalar_prefetch arg to know which expert each
tile belongs to. Print (debug) `my_expert = expert_ids_smem[bt_id]` from inside the kernel.

**Key points**:
- SMEM arrays must be **1-D** — multi-dimensional SMEM causes SC bitpacking failures.
- scalar_prefetch args go **first** in the kernel signature.
- `num_scalar_prefetch=1` in PrefetchScalarGridSpec.

In [6]:
# Exercise 0.3 — scalar_prefetch + SMEM routing table
T, D = 64, 128
BT = 8   # tile size: process BT rows per grid step

num_experts_local = 4
tokens_per_expert = T // num_experts_local  # 16

# expert_ids: (T // BT,) — which expert each tile belongs to
# (for simplicity: tile i belongs to expert i // (tiles_per_expert))
tiles_per_expert = tokens_per_expert // BT
expert_ids = jnp.repeat(jnp.arange(num_experts_local), tiles_per_expert).astype(jnp.int32)

key = jax.random.PRNGKey(0)
tokens_np = jax.random.normal(key, (T, D), dtype=jnp.bfloat16)

expert_ids, tokens_per_expert, tiles_per_expert

(Array([0, 0, 1, 1, 2, 2, 3, 3], dtype=int32), 16, 2)

In [7]:
def expert_tag_kernel(
    expert_ids_smem,   # scalar_prefetch[0]: (T//BT,) int32 — which expert per tile
    tokens_ref,
    out_ref,
):
    bt_id = pl.program_id(0)
    expert_id = expert_ids_smem[bt_id]
    jax.debug_
    
    # YOUR CODE HERE
    # 1. Read the expert ID for this tile: expert_ids_smem[bt_id]
    # 2. jax.debug.print the bt_id and expert_id
    # 3. Pass tokens through unchanged (out_ref[...] = tokens_ref[...])
    pass


# Wire up and run
# (uncomment once implemented)

In [8]:
# SOLUTION — peek if stuck
#
# Two bugs in the naive approach:
#   1. PrefetchScalarGridSpec(num_scalar_prefetch=1): BlockSpec index_map
#      receives (grid_idx, smem_ref) — not just (grid_idx).  The 1-arg lambda
#      raises "takes 1 positional argument but 2 were given".  Fix: lambda *_.
#   2. Non-trivial VMEM BlockSpec (lambda i: (i*BT,0)) causes
#      RuntimeUnexpectedCoreHalt on v4.  Fix: HBM + explicit DMA.
#   3. jax.debug.print with keyword args raises ValueError on Pallas.
#      Fix: positional args ("tile {} expert {}", bt_id, my_expert).

def _expert_tag_kernel_sol(
    expert_ids_smem,  # SMEM: (T//BT,) int32 — which expert per tile
    tokens_hbm,       # HBM: full (T, D)
    out_hbm,          # HBM: full (T, D)
    vmem_buf,         # VMEM scratch: (BT, D)
    sem_ref,          # DMA semaphore
):
    bt_id = pl.program_id(0)
    my_expert = expert_ids_smem[bt_id]
    jax.debug.print("tile {} → expert {}", bt_id, my_expert)
    dma_in  = pltpu.make_async_copy(tokens_hbm.at[pl.ds(bt_id*BT,BT),:], vmem_buf, sem_ref)
    dma_in.start(); dma_in.wait()
    dma_out = pltpu.make_async_copy(vmem_buf, out_hbm.at[pl.ds(bt_id*BT,BT),:], sem_ref)
    dma_out.start(); dma_out.wait()


def _run_expert_tag_sol(tokens, expert_ids):
    return pl.pallas_call(
        _expert_tag_kernel_sol,
        grid_spec=pltpu.PrefetchScalarGridSpec(
            num_scalar_prefetch=1,
            grid=(T // BT,),
            # HBM trivial index_map: lambda *_ to accept (grid_idx, smem_ref) from prefetch spec
            in_specs=[pl.BlockSpec((T, D), lambda *_: (0, 0), memory_space=pltpu.MemorySpace.HBM)],
            out_specs=pl.BlockSpec((T, D), lambda *_: (0, 0), memory_space=pltpu.MemorySpace.HBM),
            scratch_shapes=[pltpu.VMEM((BT, D), jnp.bfloat16), pltpu.SemaphoreType.DMA],
        ),
        out_shape=jax.ShapeDtypeStruct((T, D), jnp.bfloat16),
    )(expert_ids, tokens)


_ = _run_expert_tag_sol(tokens_np, expert_ids)
print("[PASS] expert tag kernel (check debug prints above)")


[PASS] expert tag kernel (check debug prints above)


---
# Day 1 — VMEM Compute: Single-Expert FFN (~2 hrs)

**Goal**: Write a local GEMM kernel — single expert FFN with SwiGLU activation, no communication.

This is the compute core of the full MoE kernel. Once you can do this correctly,
A2A just moves the tokens to the right device before calling this core.

## SwiGLU recap

```
gate  = token @ w_gate   # (BT, D) @ (D, F) → (BT, F)
up    = token @ w_up     # (BT, D) @ (D, F) → (BT, F)
hidden = silu(gate) * up  # SwiGLU element-wise
out   = hidden @ w_down  # (BT, F) @ (F, D) → (BT, D)
```

For the full kernel, weights are FSDP-sharded: `F_shard = F // FSDP = 128`.
Each device only holds its shard; a `psum(out, "fsdp")` outside the kernel
reduces the partial outputs.

## float32 accumulators

Always accumulate GEMMs in float32 inside VMEM, even when inputs are bfloat16.
Zero-initialize explicitly — XLA reuses freed HBM and stale values may be NaN.

## Exercise 1.1 — Local FFN kernel (no routing, no A2A)

Write a single-expert FFN kernel. For simplicity: one expert, all tokens routed to it.

Use tiles for the F dimension (bf = F_shard // 2) to stay within VMEM budget.

In [9]:
# Exercise 1.1 — Single expert SwiGLU FFN kernel

# Shapes
T1, D1, F1 = 64, 128, 128
BT1 = 8
BF1 = 64   # tile along F (must be ≥64 for bf16 128-byte alignment)
num_f_tiles = F1 // BF1

def ffn_kernel(
    tokens_ref,     # (BT1, D1)       — input tokens tile
    w_gate_ref,     # (D1, F1)         — gate weight (loaded all at once)
    w_up_ref,       # (D1, F1)         — up weight
    w_down_ref,     # (F1, D1)         — down weight
    out_ref,        # (BT1, D1)        — output
    acc_gate_ref,   # VMEM (BT1, F1) float32 — accumulator for gate
    acc_up_ref,     # VMEM (BT1, F1) float32 — accumulator for up
    acc_out_ref,    # VMEM (BT1, D1) float32 — accumulator for out
):
    # YOUR CODE HERE
    # 1. Zero-initialize all three accumulators explicitly
    # 2. For each F tile (lax.fori_loop over num_f_tiles):
    #    gate = tokens_ref @ w_gate_ref[:, f_tile*BF1 : (f_tile+1)*BF1]
    #    up   = tokens_ref @ w_up_ref[:, f_tile*BF1 : (f_tile+1)*BF1]
    #    add to acc_gate_ref, acc_up_ref
    # 3. Compute SwiGLU: hidden = silu(acc_gate_ref) * acc_up_ref
    # 4. For each D tile (lax.fori_loop over D1 // BF1):
    #    out += hidden @ w_down_ref[d_tile*BF1:(d_tile+1)*BF1, :]
    # 5. Cast acc_out_ref to bfloat16 and write to out_ref
    


def run_ffn_kernel(tokens, w_gate, w_up, w_down):
    # YOUR CODE HERE
    # Wire up pallas_call with:
    # - grid=(T1 // BT1,)
    # - in_specs: BlockSpec for tokens, load-all-rows BlockSpec for each weight
    # - scratch_shapes: 3 VMEM accumulators (float32)
    pass


def jax_ffn_reference(tokens, w_gate, w_up, w_down):
    """JAX reference for correctness checking."""
    gate = tokens.astype(jnp.float32) @ w_gate.astype(jnp.float32)
    up   = tokens.astype(jnp.float32) @ w_up.astype(jnp.float32)
    hidden = jax.nn.silu(gate) * up
    out = hidden @ w_down.astype(jnp.float32)
    return out.astype(jnp.bfloat16)


key = jax.random.PRNGKey(42)
k1, k2, k3, k4 = jax.random.split(key, 4)
tokens1  = jax.random.normal(k1, (T1, D1), dtype=jnp.bfloat16)
w_gate1  = jax.random.normal(k2, (D1, F1), dtype=jnp.bfloat16) * 0.02
w_up1    = jax.random.normal(k3, (D1, F1), dtype=jnp.bfloat16) * 0.02
w_down1  = jax.random.normal(k4, (F1, D1), dtype=jnp.bfloat16) * 0.02

ref1 = jax_ffn_reference(tokens1, w_gate1, w_up1, w_down1)
# result1 = run_ffn_kernel(tokens1, w_gate1, w_up1, w_down1)
# max_diff = jnp.max(jnp.abs(result1.astype(jnp.float32) - ref1.astype(jnp.float32)))
# print(f"max_diff={max_diff:.4e}")
# assert max_diff < 0.05, f"FFN FAIL: max_diff={max_diff}"
# print("[PASS] local FFN kernel")

In [10]:
# SOLUTION — peek if stuck
#
# Two bugs fixed vs a naive implementation on TPU v4 / JAX 0.9.2:
#
#   1. Non-trivial VMEM BlockSpec for tokens (lambda i:(i*BT1,0)) causes
#      RuntimeUnexpectedCoreHalt.  Fix: HBM BlockSpec + explicit DMA tile.
#
#   2. lax.fori_loop with dynamic index (f_t * BF1) → Mosaic alignment error
#      E2003 "cannot statically prove index is a multiple of 128".
#      Fix: Python for loops — f_start = f_t * BF1 becomes a Python int,
#      so the offset is statically known and alignment is provable.

def _ffn_kernel_sol(
    tokens_hbm, w_gate_ref, w_up_ref, w_down_ref, out_hbm,
    vmem_tok, acc_gate_ref, acc_up_ref, acc_out_ref, sem_ref,
):
    bt_id = pl.program_id(0)

    # DMA in: tokens tile
    dma_in = pltpu.make_async_copy(tokens_hbm.at[pl.ds(bt_id*BT1, BT1), :], vmem_tok, sem_ref)
    dma_in.start(); dma_in.wait()

    # Zero-initialize all float32 accumulators
    acc_gate_ref[...] = jnp.zeros_like(acc_gate_ref)
    acc_up_ref[...]   = jnp.zeros_like(acc_up_ref)
    acc_out_ref[...]  = jnp.zeros_like(acc_out_ref)

    tokens_f32 = vmem_tok[...].astype(jnp.float32)  # (BT1, D1)

    # F tiles — Python for to keep f_start static (avoids E2003 alignment error)
    for f_t in range(num_f_tiles):
        f_start = f_t * BF1                                           # Python int → static
        wg_tile = w_gate_ref[pl.ds(0, D1), pl.ds(f_start, BF1)]
        wu_tile = w_up_ref[pl.ds(0, D1), pl.ds(f_start, BF1)]
        acc_gate_ref[pl.ds(0, BT1), pl.ds(f_start, BF1)] += tokens_f32 @ wg_tile.astype(jnp.float32)
        acc_up_ref[pl.ds(0, BT1), pl.ds(f_start, BF1)]   += tokens_f32 @ wu_tile.astype(jnp.float32)

    hidden = jax.nn.silu(acc_gate_ref[...]) * acc_up_ref[...]  # (BT1, F1)

    # D tiles — same pattern
    num_d_tiles = D1 // BF1
    for d_t in range(num_d_tiles):
        d_start = d_t * BF1
        wd_tile = w_down_ref[pl.ds(0, F1), pl.ds(d_start, BF1)]
        acc_out_ref[pl.ds(0, BT1), pl.ds(d_start, BF1)] += hidden @ wd_tile.astype(jnp.float32)

    vmem_tok[...] = acc_out_ref[...].astype(jnp.bfloat16)

    # DMA out: result tile
    dma_out = pltpu.make_async_copy(vmem_tok, out_hbm.at[pl.ds(bt_id*BT1, BT1), :], sem_ref)
    dma_out.start(); dma_out.wait()


def _run_ffn_kernel_sol(tokens, w_gate, w_up, w_down):
    return pl.pallas_call(
        _ffn_kernel_sol,
        grid_spec=pltpu.PrefetchScalarGridSpec(
            num_scalar_prefetch=0,
            grid=(T1 // BT1,),
            in_specs=[
                pl.BlockSpec((T1, D1), lambda i: (0, 0), memory_space=pltpu.MemorySpace.HBM),
                pl.BlockSpec((D1, F1), lambda i: (0, 0)),
                pl.BlockSpec((D1, F1), lambda i: (0, 0)),
                pl.BlockSpec((F1, D1), lambda i: (0, 0)),
            ],
            out_specs=pl.BlockSpec((T1, D1), lambda i: (0, 0), memory_space=pltpu.MemorySpace.HBM),
            scratch_shapes=[
                pltpu.VMEM((BT1, D1), jnp.bfloat16),  # tokens DMA buffer
                pltpu.VMEM((BT1, F1), jnp.float32),   # gate accumulator
                pltpu.VMEM((BT1, F1), jnp.float32),   # up accumulator
                pltpu.VMEM((BT1, D1), jnp.float32),   # out accumulator
                pltpu.SemaphoreType.DMA,
            ],
        ),
        out_shape=jax.ShapeDtypeStruct((T1, D1), jnp.bfloat16),
    )(tokens, w_gate, w_up, w_down)


result_sol  = _run_ffn_kernel_sol(tokens1, w_gate1, w_up1, w_down1)
result_ref  = jax_ffn_reference(tokens1, w_gate1, w_up1, w_down1)
max_diff = float(jnp.max(jnp.abs(result_sol.astype(jnp.float32) - result_ref.astype(jnp.float32))))
print(f"FFN kernel max_diff={max_diff:.4e}")
assert max_diff < 0.05, f"max_diff too large: {max_diff}"
print("[PASS] FFN kernel")


FFN kernel max_diff=0.0000e+00
[PASS] FFN kernel


---
# Day 2 — Top-K Routing Without Booleans (~2 hrs)

**Goal**: Implement `get_top_k` using only int32 arithmetic — no bool arrays, no `scatter_p`.

## Why no booleans?

Mosaic has two hard restrictions:
1. **No bool reshape** — `bool_arr[:, None]` causes a shape cast Mosaic can't handle.
2. **No large 2-D bool** — `(BT, E)` bool at cluster scale (BT=8, E=256) causes
   relayout error: `"Non-singleton logical dimension is replicated in destination"`.
   Local tests with small E pass; cluster tests fail.

## The int32 row-selector pattern

Instead of `mask = (row_iota == i)  # bool`, use:
```python
row_sel = 1 - jnp.minimum(jnp.abs(row_iota - i), 1)  # int32: 1 where row==i, 0 elsewhere
```

This works because `abs(x - i)` is 0 where `x == i`, so `1 - min(0, 1) = 1`.
Elsewhere `abs ≥ 1`, so `1 - min(1, 1) = 0`.

## Why no `.at[i, :].set(v)` on JAX arrays?

`.at[i].set(v)` on a JAX array (NOT a VMEM Ref) lowers to `scatter_p`,
which is unsupported on the TPU Tensor Core path. Use `jnp.where` or the
int32 selector trick to "write" to a specific row.

Note: `.at[].set()` on a **VMEM `Ref`** (inside `pallas_call`) IS supported.

## Exercise 2.1 — get_top_k with lax.fori_loop

Implement top-K selection where:
- For each token (row), find the K highest-scoring experts
- Suppress each selected expert before finding the next
- Return `(routing: (BT, K) int32, expert_sizes: (E,) int32)`

In [ ]:
# Exercise 2.1 — get_top_k using int32 row selectors
#
# Constraints:
#   - No (BT, E) bool arrays
#   - No .at[i, :].set() on JAX arrays (scatter_p)
#   - No jnp.argmax with keepdims=True (shape cast)

BT_GK = 4   # small for easy testing
E_GK  = 8
K_GK  = 3

def get_top_k(gating, top_k):
    """
    gating: (BT_GK, E_GK) bfloat16 — router logits
    Returns:
        routing:      (BT_GK, top_k) int32  — which expert for each (token, k)
        expert_sizes: (E_GK,) int32         — how many tokens routed to each expert
    """
    bt_local, E = gating.shape
    
    # YOUR CODE HERE
    # Approach:
    # - routing: (BT_GK, top_k) int32, initially zeros
    # - hit_mask: (BT_GK, E_GK) int32 — tracks which experts have been selected
    # - For k in range(top_k):
    #   - For each row i (lax.fori_loop):
    #     * Extract row i using int32 row-selector (no bool, no scatter)
    #     * Find argmax of that row
    #     * Record in routing[i, k] using row-selector update (no scatter_p)
    #     * Mark expert as hit for row i
    #   - Suppress all hit experts (add -inf to gating)
    # - expert_sizes = sum(hit_mask, axis=0) ... but that double-counts
    #   Actually: each row selects exactly one expert per k, so
    #   expert_sizes[e] = number of (token, k) pairs where routing==e
    
    routing = jnp.zeros((bt_local, top_k), jnp.int32)
    
    # ... implement ...
    
    return routing, jnp.zeros((E,), jnp.int32)  # placeholder


# Test
key = jax.random.PRNGKey(99)
gating_test = jax.random.normal(key, (BT_GK, E_GK), dtype=jnp.bfloat16)
routing_test, sizes_test = get_top_k(gating_test, K_GK)
print("gating:\n", np.array(gating_test))
print("routing:", routing_test)
print("expert_sizes:", sizes_test)

# Verify: no duplicates per row
for row in range(BT_GK):
    experts = set(int(routing_test[row, k]) for k in range(K_GK))
    assert len(experts) == K_GK, f"Row {row} has duplicates: {experts}"
print("[no duplicates check passed]")

In [ ]:
# SOLUTION — peek if stuck

def _get_top_k_sol(gating, top_k):
    bt_local, E = gating.shape
    row_iota_e  = lax.broadcasted_iota(jnp.int32, gating.shape, 0)      # (BT, E) — row idx
    row_iota_k  = lax.broadcasted_iota(jnp.int32, (bt_local, top_k), 0) # (BT, K) — row idx
    col_iota_k  = lax.broadcasted_iota(jnp.int32, (bt_local, top_k), 1) # (BT, K) — col idx
    expert_iota = lax.broadcasted_iota(jnp.int32, (bt_local, E), 1)     # (BT, E) — expert idx

    routing      = jnp.zeros((bt_local, top_k), jnp.int32)
    all_hit_masks = jnp.zeros_like(gating, dtype=jnp.int32)  # (BT, E)

    for k in range(top_k):
        def pick_row(i, carry):
            routing_k, hit_mask_k, inp = carry
            # Row selector: 1 at row i, 0 elsewhere
            row_sel = 1 - jnp.minimum(jnp.abs(row_iota_e - i), 1)  # (BT, E)
            # Extract row i: sum over rows (only row i contributes)
            row_f32 = jnp.sum(
                inp.astype(jnp.float32) * row_sel.astype(jnp.float32), axis=0
            )  # (E,)
            best = jnp.argmax(row_f32).astype(jnp.int32)  # scalar

            # Update routing[i, k]: use BOTH row AND column selectors so only
            # cell (i, k) is written.  Without col_sel_k every column of row i
            # gets overwritten to `best`, clobbering earlier k selections.
            row_sel_k = 1 - jnp.minimum(jnp.abs(row_iota_k - i), 1)   # (BT,K): 1 at row i
            col_sel_k = 1 - jnp.minimum(jnp.abs(col_iota_k - k), 1)   # (BT,K): 1 at col k
            cell_sel  = row_sel_k * col_sel_k                           # (BT,K): 1 at (i,k)
            routing_k = routing_k * (1 - cell_sel) + cell_sel * best

            # Mark expert 'best' as hit for row i
            expert_hit = (expert_iota[i] == best).astype(jnp.int32)  # (E,)
            hit_update = row_sel * expert_hit[None, :]  # (BT, E): 1 only at (i, best)
            hit_mask_k = hit_mask_k + hit_update
            return routing_k, hit_mask_k, inp

        init_hit = jnp.zeros_like(gating, dtype=jnp.int32)
        routing, hit_mask_k, gating = lax.fori_loop(
            0, bt_local, pick_row, (routing, init_hit, gating)
        )
        all_hit_masks = all_hit_masks + hit_mask_k
        # Suppress selected experts
        _neg_inf = jnp.array(jnp.finfo(gating.dtype).min, dtype=gating.dtype)
        gating = gating + hit_mask_k.astype(gating.dtype) * _neg_inf

    expert_sizes = jnp.sum(all_hit_masks, axis=0)  # (E,)
    return routing, expert_sizes


routing_sol, sizes_sol = _get_top_k_sol(gating_test, K_GK)
print("routing (sol):", routing_sol)
print("expert_sizes (sol):", sizes_sol)

# Verify: no duplicates per row
for row in range(BT_GK):
    experts = set(int(routing_sol[row, k]) for k in range(K_GK))
    assert len(experts) == K_GK, f"Row {row} has duplicates: {experts}"

# Verify: sizes sum matches BT*K
assert int(jnp.sum(sizes_sol)) == BT_GK * K_GK
print("[PASS] get_top_k")


---
# Day 3 — ICI Primitives: Ring-Reduce Metadata (~2 hrs)

**Goal**: Understand ICI remote copy and implement a metadata ring-reduce over EP devices.

## Why metadata ring-reduce?

Before A2A scatter, each device knows how many of its own tokens go to each expert
(`local_expert_sizes: (E,)`). But to compute where in the receive buffer to write
incoming tokens, it needs the **global** per-expert counts — aggregated across all EP devices.

A ring-reduce passes each device's row around the ring, accumulating as it goes.
After `EP-1` steps, every device has the full aggregated metadata.

## ICI remote copy

```python
pltpu.async_remote_copy(
    src_ref=local_hbm.at[pl.ds(src_off, size)],
    dst_ref=remote_hbm.at[pl.ds(dst_off, size)],  # on target device's HBM
    send_sem=send_sem,   # signaled on SENDER when data left
    recv_sem=recv_sem,   # signaled on RECEIVER when data arrived
    device_id=target_device_id,
    device_id_type=pltpu.DeviceIdType.MESH,
).start()
# Then: on sender: wait(send_sem, 1) to know src buffer is reusable
#       on receiver: wait(recv_sem, 1) to know data arrived
```

**CRITICAL**: receiver waits on `recv_sem`, sender waits on `send_sem`.
The receiver calls `pltpu.semaphore_wait(recv_sem, count=1)` — NOT the sender.

## Barrier semaphore

Before each ring step, all devices must have written their data.
Use `pltpu.get_barrier_semaphore()` + signal + wait to synchronize globally.

## Exercise 3.1 — simulate ring-reduce in JAX (no Pallas yet)

Before writing Pallas code, implement the ring-reduce logic in pure JAX using `lax.fori_loop`.
This verifies the algorithm before dealing with DMA semantics.

**Input**: `local_counts: (EP, E)` — device `i` holds row `i` = its local expert sizes.
**Output**: `(global_sizes: (E,), my_start_offsets: (E,))`
- `global_sizes[e]` = total tokens routed to expert `e` across all devices
- `my_start_offsets[e]` = this device's write offset in expert `e`'s receive buffer

In [ ]:
# Exercise 3.1 — ring-reduce metadata (JAX simulation)
#
# Simulates EP devices doing a ring-reduce over expert sizes.
# For device my_id: starts with local_counts[my_id], receives rows from left neighbor.

EP_RR = 4
E_RR  = 8

def ring_reduce_one_device(local_counts, my_id, EP, E):
    """
    local_counts: (EP, E) int32 — all devices' counts (simulating shared HBM)
    my_id: int — this device's index
    Returns: (global_sizes: (E,), my_start_offsets: (E,))
    """
    # YOUR CODE HERE
    # Algorithm:
    # - acc_sizes  = local_counts[my_id]  (my local counts)
    # - acc_starts = zeros
    # - For step in range(EP - 1):
    #   - row_to_read = (my_id - step - 1) % EP  (the row sent to me in this step)
    #   - new_counts = local_counts[row_to_read]
    #   - acc_sizes += new_counts
    #   - if (row_to_read < my_id): acc_starts += new_counts
    #     (only accumulate offset for devices that come before me in expert ordering)
    # Return (acc_sizes, acc_starts)
    pass


# Test: random counts
key = jax.random.PRNGKey(7)
counts = jax.random.randint(key, (EP_RR, E_RR), 0, 10, dtype=jnp.int32)
print("counts per device:\n", np.array(counts))

# For each device, compute ring-reduce
for dev in range(EP_RR):
    sizes, starts = ring_reduce_one_device(counts, dev, EP_RR, E_RR)
    expected_sizes = jnp.sum(counts, axis=0)
    # All devices should agree on global sizes
    if sizes is not None:
        assert jnp.all(sizes == expected_sizes), f"dev {dev}: sizes mismatch"
        # Start offsets: sum of counts[j] for j < dev
        expected_starts = jnp.sum(counts[:dev], axis=0)
        assert jnp.all(starts == expected_starts), f"dev {dev}: starts mismatch, got {starts}, expected {expected_starts}"
        print(f"dev {dev}: sizes={np.array(sizes)[:4]}... starts={np.array(starts)[:4]}... [OK]")

In [ ]:
# SOLUTION — peek if stuck

def _ring_reduce_one_device_sol(local_counts, my_id, EP, E):
    acc_sizes  = local_counts[my_id].astype(jnp.int32)
    acc_starts = jnp.zeros((E,), jnp.int32)

    def ring_step(step, carry):
        acc_s, acc_st = carry
        # Which row arrives at me in this step
        row_id = (my_id - step - 1) % EP
        new_counts = local_counts[row_id]
        acc_s = acc_s + new_counts
        # Add to starts only if row_id < my_id (this device's tokens come before mine)
        add_to_start = jnp.where(row_id < my_id, new_counts, jnp.zeros_like(new_counts))
        acc_st = acc_st + add_to_start
        return acc_s, acc_st

    acc_sizes, acc_starts = lax.fori_loop(0, EP - 1, ring_step, (acc_sizes, acc_starts))
    return acc_sizes, acc_starts


for dev in range(EP_RR):
    sizes, starts = _ring_reduce_one_device_sol(counts, dev, EP_RR, E_RR)
    expected_sizes  = jnp.sum(counts, axis=0)
    expected_starts = jnp.sum(counts[:dev], axis=0)
    assert jnp.all(sizes  == expected_sizes),  f"dev {dev}: sizes mismatch"
    assert jnp.all(starts == expected_starts), f"dev {dev}: starts mismatch"
    print(f"dev {dev}: [OK]")
print("[PASS] ring-reduce simulation")

---
# Day 4 — A2A Scatter + Gather (~2 hrs)

**Goal**: Implement the ICI A2A logic — send tokens to expert-owning devices, receive back.

## EP MoE data flow

```
Each device owns:
  tokens[my_id]:   (T_fsdp, D)    — my local tokens
  experts[my_id]:  E_local experts — I compute these

Scatter phase: for each (token t, expert e in routing[t]):
  if expert e lives on device d_e != my device:
    send tokens[t] to d_e's receive buffer

GEMM phase: run FFN on all tokens received for my local experts

Gather phase: send computed results back to token-owning devices
```

## Exercise 4.1 — A2A scatter/gather in JAX (simulation, no Pallas)

Simulate EP=4 devices doing A2A. Each device has 4 tokens, 2 local experts.
After scatter+GEMM+gather, verify the output matches a global (non-distributed) reference.

This is the algorithmic skeleton that the Pallas kernel implements with async_remote_copy.

In [ ]:
# Exercise 4.1 — EP MoE A2A simulation in Python
#
# We simulate 4 devices, each with 4 tokens, 2 experts (8 total), K=2 routing.

EP_A2A = 4
E_A2A  = 8   # total experts
E_local_A2A = E_A2A // EP_A2A  # = 2
T_local_A2A = 4   # tokens per device
D_A2A = 16
F_A2A = 8
K_A2A = 2

# Generate data
key = jax.random.PRNGKey(123)
keys = jax.random.split(key, 10)
all_tokens  = jax.random.normal(keys[0], (EP_A2A, T_local_A2A, D_A2A), dtype=jnp.float32)
all_w_gate  = jax.random.normal(keys[1], (E_A2A, D_A2A, F_A2A), dtype=jnp.float32) * 0.1
all_w_up    = jax.random.normal(keys[2], (E_A2A, D_A2A, F_A2A), dtype=jnp.float32) * 0.1
all_w_down  = jax.random.normal(keys[3], (E_A2A, F_A2A, D_A2A), dtype=jnp.float32) * 0.1
all_gating  = jax.random.normal(keys[4], (EP_A2A, T_local_A2A, E_A2A), dtype=jnp.float32)

# Pre-compute routing (use simple argmax top-K for simulation clarity)
def simple_top_k(gating_row, k):
    """Returns top-k expert indices for one token."""
    return jnp.argsort(-gating_row)[:k]

# all_routing[dev, token, k] = expert index
all_routing = np.zeros((EP_A2A, T_local_A2A, K_A2A), dtype=np.int32)
for dev in range(EP_A2A):
    for t in range(T_local_A2A):
        all_routing[dev, t] = simple_top_k(all_gating[dev, t], K_A2A)

# Routing weights (softmax of top-k logits, renormalized)
def routing_weights(gating_row, routing_k):
    scores = gating_row[routing_k]
    weights = jax.nn.softmax(scores)
    return weights

all_rw = np.zeros((EP_A2A, T_local_A2A, K_A2A), dtype=np.float32)
for dev in range(EP_A2A):
    for t in range(T_local_A2A):
        all_rw[dev, t] = routing_weights(all_gating[dev, t], all_routing[dev, t])


def run_ep_moe_simulation(dev_id):
    """
    Run EP MoE forward for device dev_id.
    Returns output: (T_local_A2A, D_A2A)
    """
    # YOUR CODE HERE
    # Steps:
    # 1. Scatter: collect tokens from all devices for my experts
    #    - For each device d, for each token t, for each k:
    #      if all_routing[d, t, k] // E_local_A2A == dev_id:
    #        add all_tokens[d, t] to my_received[e_local, list]
    #
    # 2. GEMM: run FFN for each of my local experts
    #    - For each e_local in range(E_local_A2A):
    #      e_global = dev_id * E_local_A2A + e_local
    #      for each token received for this expert: compute FFN output
    #
    # 3. Gather: collect FFN outputs back to token-owning devices
    #    - Build output by summing weighted expert outputs for my T_local_A2A tokens
    #
    # Return the (T_local_A2A, D_A2A) output for dev dev_id
    pass


def run_global_reference(dev_id):
    """Non-distributed reference: just compute MoE for dev_id's tokens."""
    out = np.zeros((T_local_A2A, D_A2A), dtype=np.float32)
    for t in range(T_local_A2A):
        token = np.array(all_tokens[dev_id, t])  # (D,)
        for k in range(K_A2A):
            e = all_routing[dev_id, t, k]
            wg = np.array(all_w_gate[e])
            wu = np.array(all_w_up[e])
            wd = np.array(all_w_down[e])
            gate   = token @ wg
            up     = token @ wu
            hidden = 1 / (1 + np.exp(-gate)) * gate * up  # silu * up
            out[t] += all_rw[dev_id, t, k] * (hidden @ wd)
    return out


# Test
for dev in range(EP_A2A):
    result = run_ep_moe_simulation(dev)
    ref = run_global_reference(dev)
    if result is not None:
        max_diff = np.max(np.abs(result - ref))
        print(f"dev {dev}: max_diff={max_diff:.4e}")
        assert max_diff < 1e-4, f"FAIL dev {dev}: max_diff={max_diff}"
print("[All devices pass]")

In [ ]:
# SOLUTION — peek if stuck

def _run_ep_moe_simulation_sol(dev_id):
    # Step 1: Scatter — collect tokens from all devices for my experts
    received = {e_local: [] for e_local in range(E_local_A2A)}

    for src_dev in range(EP_A2A):
        for t in range(T_local_A2A):
            for k in range(K_A2A):
                e_global = all_routing[src_dev, t, k]
                owner = e_global // E_local_A2A
                if owner == dev_id:
                    e_local = e_global % E_local_A2A
                    received[e_local].append((
                        np.array(all_tokens[src_dev, t]),
                        all_rw[src_dev, t, k],
                        src_dev, t, k
                    ))

    # Step 2: GEMM — run FFN for each local expert
    expert_outputs = {}
    for e_local in range(E_local_A2A):
        e_global = dev_id * E_local_A2A + e_local
        wg = np.array(all_w_gate[e_global])
        wu = np.array(all_w_up[e_global])
        wd = np.array(all_w_down[e_global])
        for (token_vec, weight, src_dev, t, k) in received[e_local]:
            gate   = token_vec @ wg
            up     = token_vec @ wu
            hidden = 1 / (1 + np.exp(-gate)) * gate * up
            out_vec = hidden @ wd
            expert_outputs[(src_dev, t, k)] = (out_vec, weight)

    # Step 3: Gather — accumulate weighted expert outputs for MY tokens.
    # expert_outputs only has entries for tokens routed to experts on THIS device.
    # For tokens whose expert lives on another device we must compute the FFN
    # output directly (in a real system this would arrive via the A2A gather).
    out = np.zeros((T_local_A2A, D_A2A), dtype=np.float32)
    for t in range(T_local_A2A):
        for k in range(K_A2A):
            key_dtk = (dev_id, t, k)
            if key_dtk in expert_outputs:
                out_vec, weight = expert_outputs[key_dtk]
                out[t] += weight * out_vec
            else:
                # Expert is on another device — compute FFN directly
                e_global = int(all_routing[dev_id, t, k])
                weight   = float(all_rw[dev_id, t, k])
                wg = np.array(all_w_gate[e_global])
                wu = np.array(all_w_up[e_global])
                wd = np.array(all_w_down[e_global])
                token_vec = np.array(all_tokens[dev_id, t])
                gate   = token_vec @ wg
                up     = token_vec @ wu
                hidden = 1 / (1 + np.exp(-gate)) * gate * up
                out_vec = hidden @ wd
                out[t] += weight * out_vec
    return out


for dev in range(EP_A2A):
    result = _run_ep_moe_simulation_sol(dev)
    ref = run_global_reference(dev)
    max_diff = np.max(np.abs(result - ref))
    print(f"dev {dev}: max_diff={max_diff:.4e}")
    assert max_diff < 1e-4, f"FAIL dev {dev}: max_diff={max_diff}"
print("[PASS] A2A simulation")


---
# Day 5 — Full Kernel Skeleton + AOT Compile Check (~2 hrs)

**Goal**: Wire all phases into a single Pallas kernel body and run an AOT compile check
using virtual v7x devices (no hardware needed).

## What you have so far

- Day 0: `pallas_call` structure, BlockSpec, DMA semaphores
- Day 1: VMEM GEMM with float32 accumulators, SwiGLU activation
- Day 2: `get_top_k` with int32 row-selectors (no booleans)
- Day 3: Ring-reduce algorithm in `lax.fori_loop`
- Day 4: A2A scatter/gather algorithm

## Integration challenges

Putting it all together inside a Pallas kernel body adds new constraints:

1. **SMEM arrays must be 1-D** — routing tables, counts, offsets all 1-D.
2. **ICI requires `collective_id`** — set in `pltpu.CompilerParams(collective_id=N)`.
   Different call sites (primal vs VJP forward) need different IDs.
3. **Device ID from `lax.axis_index`** — call OUTSIDE the kernel (at shard_map level),
   pass in via `scalar_prefetch`. Inside the kernel, `tp_rank_scalar[0]` gives the value.
4. **No `@jax.jit` on any function called inside `shard_map`** — causes 10×+ compile slowdown.
5. **`lax.fori_loop` not Python loop** — Python loops unroll at trace time (compile OOM/timeout
   for large EP). Use `lax.fori_loop` for all iteration over EP devices.

## Exercise 5.1 — Assemble the kernel skeleton

Wire together the phases. You don't need to have every line correct — the goal
is to get the Pallas kernel structure right so the AOT compile check (Exercise 5.2)
can validate the shapes and SMEM layout.

In [ ]:
# Exercise 5.1 — Full kernel skeleton
#
# Fill in the phase calls. The actual implementation of each phase
# comes from the helpers we built on Days 0-4.
#
# This is the real production kernel shape for EP=32, FSDP=16, TP=1.

# Config matching bodaborg 4x4x16, v205
EP_FULL = 32
FSDP_FULL = 16
E_FULL = 256
E_local_FULL = E_FULL // EP_FULL   # = 8
D_FULL = 7168
F_shard_FULL = 128     # F // FSDP = 2048 // 16
K_FULL = 8
BT_FULL = 8            # token tile size


def _moe_kernel_skeleton(
    # scalar_prefetch (loaded to SMEM before each grid step)
    tp_rank_scalar,        # (1,) int32 — from lax.axis_index("tp") at shard_map level
    # HBM inputs (BlockSpec tells Pallas how to slice each step)
    tokens_ref,            # (BT_FULL, D_FULL) bfloat16
    w1_ref,                # (E_local_FULL, 2, D_FULL, F_shard_FULL) — gate+up stacked
    w2_ref,                # (E_local_FULL, F_shard_FULL, D_FULL)
    gating_ref,            # (BT_FULL, E_FULL) bfloat16
    # HBM scratch (allocated by pallas_call, shape passed as out_shape list)
    a2a_scatter_buf_ref,   # (EP_FULL * BT_FULL, D_FULL) — A2A receive buffer
    a2a_gather_buf_ref,    # (E_local_FULL, BT_FULL, D_FULL) — gather output buffer
    # HBM output
    out_ref,               # (BT_FULL, D_FULL) bfloat16
    # SMEM scratch (ALL 1-D — mandatory for E=256)
    routing_smem,          # (BT_FULL * K_FULL,) int32 — t2e_routing flattened
    d2e_count_smem,        # (EP_FULL * E_FULL,) int32 — ring-reduce metadata
    expert_starts_smem,    # (E_FULL,) int32 — global offset per expert
    expert_sizes_smem,     # (E_FULL,) int32 — global count per expert
    # VMEM scratch
    gating_vmem,           # (BT_FULL, E_FULL) float32
    acc_gate_vmem,         # (BT_FULL, F_shard_FULL) float32 — GEMM accumulator
    acc_up_vmem,           # (BT_FULL, F_shard_FULL) float32
    acc_out_vmem,          # (BT_FULL, D_FULL) float32
    # Semaphores
    local_sems,            # DMA — for local HBM↔VMEM copies
    send_sems,             # DMA — ICI send semaphores
    recv_sems,             # DMA — ICI receive semaphores
    *,
    top_k: int,
    ep_axis_name: str,
):
    bt_id = pl.program_id(0)
    my_id = lax.axis_index(ep_axis_name).astype(jnp.int32)
    num_devices = lax.axis_size(ep_axis_name)
    fsdp_rank = lax.axis_index("fsdp").astype(jnp.int32)

    def get_mesh_device_id(ep_rank):
        # (dp=0, ep=ep_rank, fsdp=fsdp_rank, tp=0) for TP=1
        return (jnp.int32(0), ep_rank, fsdp_rank, tp_rank_scalar[0])

    # YOUR CODE HERE
    # Phase 0: Top-K routing
    #   - Load gating from gating_ref into gating_vmem
    #   - Call get_top_k to get (routing: BT×K, expert_sizes: E)
    #   - Store routing into routing_smem (flattened 1-D)

    # Phase 1: Metadata ring-reduce
    #   - Store my expert_sizes into d2e_count_smem[my_id * E : (my_id+1) * E]
    #   - Ring-reduce to compute global sizes and my start offsets
    #   - Load results into expert_starts_smem, expert_sizes_smem

    # Phase 2: A2A scatter
    #   - For each expert e (lax.fori_loop over E_FULL):
    #     owner = e // E_local_FULL
    #     start = expert_starts_smem[e]  (where this device's tokens go)
    #     async_remote_copy tokens → a2a_scatter_buf_ref on owner device

    # Phase 3: GEMM
    #   - Wait for all tokens to arrive (recv_sems)
    #   - For each local expert e_local:
    #     run FFN with double-buffered weight loads

    # Phase 4: A2A gather
    #   - Send computed results from a2a_gather_buf_ref back to token owners

    # Phase 5: Accumulate
    #   - Wait for all gather results
    #   - Apply routing weights, sum into out_ref

    pass   # placeholder

## Exercise 5.2 — AOT Compile Check (virtual v7x, no hardware)

Before ever touching the cluster, run an AOT compile check using `jax.experimental.topologies`.
This gives you real Mosaic compilation (v7x backend) in ~3-5 minutes on any machine
with `libtpu` in the venv.

**What this catches**:
- Wrong SMEM dimensionality (should be 1-D)
- Trailing size-1 VMEM dimensions
- Bool reshape or large 2-D bool
- `scatter_p` on JAX arrays
- Shape mismatches in BlockSpec

**What it does NOT catch**: runtime bugs (wrong routing, wrong offsets, NaN from wrong accumulation)

After AOT passes, run the test ladder:
1. EP=1 execution test (single chip, full compute)
2. EP=4 execution test (A2A over 4 chips)
3. Full cluster (EP=32, FSDP=16 on bodaborg)

In [ ]:
# Exercise 5.2 — AOT compile check
#
# This uses virtual devices — no physical TPU needed.
# Requires libtpu in the venv: source ~/xdb/.xprof/bin/activate

from jax.experimental import topologies

def run_aot_check():
    # YOUR CODE HERE
    # 1. Get virtual 4x4x16 topology (512 virtual v7x devices)
    #    topo = topologies.get_topology_desc("tpu7x:4x4x16", platform="tpu")
    #
    # 2. Build mesh: (dp=1, ep=32, fsdp=16, tp=1)
    #    devs = np.array(topo.devices).reshape(1, EP_FULL, FSDP_FULL, 1)
    #    mesh = jax.sharding.Mesh(devs, ("dp", "ep", "fsdp", "tp"))
    #
    # 3. Define the shard_map body that calls your kernel
    #    (mirror the structure from fwd_kernel_train.py)
    #
    # 4. Create abstract input shapes (jax.ShapeDtypeStruct)
    #
    # 5. jit.lower(...).compile() — if it raises, fix the shape error
    pass


# Uncomment to run (takes 3-5 minutes)
# run_aot_check()

In [ ]:
# SOLUTION — AOT check skeleton (mirrors fused_moe_bwd/test_fwd_train_aot.py)

def _run_aot_check_sol():
    from jax.experimental import topologies

    DP, EP, FSDP, TP = 1, 32, 16, 1
    GBS, SEQ = 256, 4096
    D, F, E, K = 7168, 2048, 256, 8
    TOTAL = DP * EP * FSDP * TP  # 512

    print(f"[AOT] Loading tpu7x:4x4x16 virtual topology ({TOTAL} devices)...")
    topo = topologies.get_topology_desc("tpu7x:4x4x16", platform="tpu")
    assert len(topo.devices) == TOTAL

    devs = np.array(topo.devices).reshape(DP, EP, FSDP, TP)
    mesh = jax.sharding.Mesh(devs, ("dp", "ep", "fsdp", "tp"))
    print(f"[AOT] Mesh: {dict(mesh.shape)}")

    T_total = GBS * SEQ         # 1_048_576
    T_fsdp  = T_total // FSDP   # 65_536

    # Inside shard_map:
    #   tokens: (T_fsdp, D), w1: (E//EP, 2, D, F//FSDP), w2: (E//EP, F//FSDP, D)
    #   gating: (T_fsdp, E)

    def kernel_body(tokens, w1, w2, gating, _tp_dummy):
        # tp_rank from lax.axis_index at shard_map level (partition_id)
        tp_rank = lax.axis_index("tp").astype(jnp.int32)
        tp_rank_arr = jnp.full((1,), tp_rank, dtype=jnp.int32)

        # Call your kernel here — _moe_kernel_skeleton(...)
        # For AOT check, return a placeholder shape
        return jnp.zeros_like(tokens)

    sharded_fn = functools.partial(
        shard_map,
        f=kernel_body,
        mesh=mesh,
        in_specs=(
            P("fsdp", None),        # tokens
            P("ep", None, None, "fsdp"),  # w1: (E//EP, 2, D, F//FSDP)
            P("ep", "fsdp", None),  # w2: (E//EP, F//FSDP, D)
            P("fsdp", None),        # gating
            P("tp"),                # tp_dummy — brings tp into scope
        ),
        out_specs=P("fsdp", None),
        check_rep=False,
    )()

    tokens_abs = jax.ShapeDtypeStruct((T_total, D), jnp.bfloat16)
    w1_abs     = jax.ShapeDtypeStruct((E, 2, D, F // FSDP), jnp.bfloat16)
    w2_abs     = jax.ShapeDtypeStruct((E, F // FSDP, D), jnp.bfloat16)
    gating_abs = jax.ShapeDtypeStruct((T_total, E), jnp.bfloat16)
    tp_dummy   = jax.ShapeDtypeStruct((TP,), jnp.int32)

    print("[AOT] Lowering...")
    with jax.default_device(topo.devices[0]):
        lowered = jax.jit(sharded_fn).lower(
            tokens_abs, w1_abs, w2_abs, gating_abs, tp_dummy
        )

    print("[AOT] Compiling (Mosaic, ~3-5 min)...")
    try:
        compiled = lowered.compile()
        print("[AOT] PASS")
    except Exception as e:
        print(f"[AOT] FAIL:\n{e}")
        raise


# Uncomment to run (requires libtpu in xprof venv, ~3-5 min)
# _run_aot_check_sol()

---
# Day 6 — Correctness: EP=1 Execution Test (~2 hrs)

**Goal**: Run the complete kernel end-to-end on a single physical device and
compare output to the JAX reference.

## EP=1 simplifications

With EP=1, there is no A2A communication. Every token goes to a local expert.
This is the simplest possible configuration to validate correctness:
- No ring-reduce (only one device, nothing to aggregate)
- No ICI scatter/gather (all tokens stay local)
- GEMM phase is identical to full config

Once EP=1 passes, you can test EP=4 to validate the A2A paths.

## Exercise 6.1 — EP=1 correctness test

Run your kernel (or the reference kernel `fused_ep_moe_fwd_train_v1`) with EP=1
and compare against `jax.vmap` over experts.

In [ ]:
# Exercise 6.1 — EP=1 correctness test
#
# This runs on your local machine's JAX devices (or a single TPU chip)
# No cluster needed.

# Small config for local testing
T_EP1, D_EP1, F_EP1 = 256, 128, 64
E_EP1, K_EP1 = 8, 2
E_local_EP1 = E_EP1  # EP=1: all experts local

def jax_moe_reference_ep1(tokens, w_gate, w_up, w_down, gating, K):
    """
    Pure JAX reference for MoE forward pass.
    tokens:  (T, D)
    w_gate:  (E, D, F)
    w_up:    (E, D, F)
    w_down:  (E, F, D)
    gating:  (T, E)
    Returns: (T, D)
    """
    T, D = tokens.shape
    # Top-K routing
    top_k_experts = jnp.argsort(-gating, axis=-1)[:, :K]  # (T, K)
    top_k_scores  = jnp.take_along_axis(gating, top_k_experts, axis=-1)  # (T, K)
    top_k_weights = jax.nn.softmax(top_k_scores, axis=-1)  # (T, K) normalized

    out = jnp.zeros((T, D), dtype=jnp.float32)

    # YOUR CODE HERE
    # For each (t, k): compute FFN output for expert top_k_experts[t, k],
    # weight by top_k_weights[t, k], accumulate into out[t]
    # Hint: use a Python double loop for clarity — correctness not speed

    return out.astype(jnp.bfloat16)


# Generate test data
key = jax.random.PRNGKey(456)
k1, k2, k3, k4, k5 = jax.random.split(key, 5)
tok_ep1   = jax.random.normal(k1, (T_EP1, D_EP1), jnp.bfloat16)
wg_ep1    = jax.random.normal(k2, (E_EP1, D_EP1, F_EP1), jnp.bfloat16) * 0.01
wu_ep1    = jax.random.normal(k3, (E_EP1, D_EP1, F_EP1), jnp.bfloat16) * 0.01
wd_ep1    = jax.random.normal(k4, (E_EP1, F_EP1, D_EP1), jnp.bfloat16) * 0.01
gate_ep1  = jax.random.normal(k5, (T_EP1, E_EP1), jnp.bfloat16)

ref_out = jax_moe_reference_ep1(tok_ep1, wg_ep1, wu_ep1, wd_ep1, gate_ep1, K_EP1)
print(f"Reference output shape: {ref_out.shape}, range: [{float(ref_out.min()):.3f}, {float(ref_out.max()):.3f}]")

# Once your kernel is ready:
# kernel_out = your_kernel(tok_ep1, wg_ep1, wu_ep1, wd_ep1, gate_ep1, K_EP1)
# max_diff = jnp.max(jnp.abs(kernel_out.astype(jnp.float32) - ref_out.astype(jnp.float32)))
# assert max_diff < 0.02, f"EP=1 FAIL: max_diff={max_diff}"
# print(f"[PASS] EP=1 correctness: max_diff={max_diff:.4e}")

In [ ]:
# SOLUTION — JAX reference

def _jax_moe_reference_ep1_sol(tokens, w_gate, w_up, w_down, gating, K):
    T, D = tokens.shape
    E, _, F = w_gate.shape

    top_k_experts = jnp.argsort(-gating, axis=-1)[:, :K]  # (T, K)
    top_k_scores  = jnp.take_along_axis(gating, top_k_experts, axis=-1)
    top_k_weights = jax.nn.softmax(top_k_scores.astype(jnp.float32), axis=-1)

    tokens_f32 = tokens.astype(jnp.float32)
    w_gate_f32 = w_gate.astype(jnp.float32)
    w_up_f32   = w_up.astype(jnp.float32)
    w_down_f32 = w_down.astype(jnp.float32)

    out = jnp.zeros((T, D), dtype=jnp.float32)
    for t in range(T):
        for k in range(K):
            e = int(top_k_experts[t, k])
            w_k = float(top_k_weights[t, k])
            tok = tokens_f32[t]   # (D,)
            gate   = tok @ w_gate_f32[e]   # (F,)
            up     = tok @ w_up_f32[e]     # (F,)
            hidden = jax.nn.silu(gate) * up  # (F,)
            ffn_out = hidden @ w_down_f32[e]  # (D,)
            out = out.at[t].add(w_k * ffn_out)

    return out.astype(jnp.bfloat16)


ref_sol = _jax_moe_reference_ep1_sol(tok_ep1, wg_ep1, wu_ep1, wd_ep1, gate_ep1, K_EP1)
print(f"Reference output: {ref_sol.shape}, range [{float(ref_sol.min()):.4f}, {float(ref_sol.max()):.4f}]")
# Compare with your kernel output once implemented
print("[INFO] JAX reference ready — compare against your kernel output")

---
# Appendix: Key Mosaic Constraints Reference

Quick-reference for all hard constraints. Each has caused silent bugs or compile
failures in production — treat as invariants.

| Constraint | Wrong | Right |
|---|---|---|
| SMEM must be 1-D | `SMEM((E, K), int32)` | `SMEM((E*K,), int32)` |
| No trailing size-1 VMEM | `VMEM((A, B, 1, N), bf16)` | `VMEM((A, B, N), bf16)` |
| No bool reshape | `mask[:, None]` | build 2-D int32 explicitly |
| No large 2-D bool | `(bt, E) == i` → bool | int32 row selector |
| No scatter on JAX arrays | `arr.at[i].set(v)` | `jnp.where(mask, v, arr)` |
| No `@jax.jit` in shard_map | decorate fn inside shard_map | remove decorator |
| lax.fori_loop not Python loop | `for e in range(EP)` | `lax.fori_loop(0, EP, fn, init)` |
| Zero-init VMEM accumulators | rely on XLA to zero | `acc[...] = jnp.zeros_like(acc)` |
| tp_rank via lax.axis_index | `tp_iota[0]` (SPMD folds to 0) | `lax.axis_index("tp")` |
| Distinct collective_id per call | both uses share id=0 | primal=0, VJP fwd=1 |
| remat + Pallas A2A | default policy re-runs DMA | `save_only_these_names('pallas_moe_out')` |

---

## Test Ladder (never skip levels)

1. **AOT compile check** (Day 5): virtual v7x devices, Mosaic compile, no hardware, ~3-5 min
2. **EP=1 execution test** (Day 6): single chip, full compute, checks routing + GEMM math
3. **EP=4 execution test**: 4 chips, checks A2A scatter/gather paths
4. **Full cluster**: EP=32, FSDP=16 on bodaborg 4×4×16, checks ICI ring-reduce + A2A at scale

Mini configs (EP=1, FSDP=1) hide FSDP bugs because `F_shard == F_full`. Always test at
scale before declaring correctness.